# LeetCode 1653 — Minimum Deletions to Make String Balanced bằng Python

Link đề: https://leetcode.com/problems/minimum-deletions-to-make-string-balanced/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Stack

Skeleton LeetCode:

```python
class Solution:
    def minimumDeletions(self, s: str) -> int:
        ...
```

Notebook đi từ **thử mọi tập con để xóa** (định nghĩa) → **thử mọi điểm cắt** `O(n²)` → **tiền tố/hậu tố** `O(n)` → **hai biến đếm** `O(1)` bộ nhớ → **quy hoạch động** một biến → **ngăn xếp** triệt tiêu cặp `"ba"`.

## 1. Mục tiêu học

1. Hiểu "cân bằng" ⇔ xâu có dạng `a…ab…b` (mọi `a` đứng trước mọi `b`), nên lời giải tương ứng với việc chọn một **điểm cắt** `k`.
2. Đếm chi phí của một điểm cắt bằng **tiền tố đếm `b`** và **hậu tố đếm `a`**, rồi tối ưu xuống hai biến chạy.
3. Xây dựng **quy hoạch động** `f[i]` = số xóa ít nhất cho tiền tố dài `i`, với lựa chọn "xóa `a` này" hay "xóa mọi `b` trước nó".
4. Hiểu lời giải **ngăn xếp**: mỗi lần gặp cặp `b…a` "chưa ghép" thì xóa một ký tự — và vì sao nó tối ưu.
5. Đối chiếu tất cả các cách với vét cạn tập con trên xâu ngắn, seed cố định.

## 2. Đề bài & ví dụ

Cho xâu `s` chỉ gồm `'a'` và `'b'`. Bạn được **xóa** bao nhiêu ký tự tùy ý để `s` trở nên **cân bằng**. Xâu `s` cân bằng nếu **không** tồn tại cặp chỉ số `(i, j)` với `i < j`, `s[i] = 'b'` và `s[j] = 'a'`.

Trả về **số lần xóa ít nhất** để `s` cân bằng.

**Ví dụ 1:** `s = "aababbab"` → `2`. Có thể xóa vị trí 2 và 6 (`"aababbab"` → `"aaabbb"`) hoặc vị trí 3 và 6 (→ `"aabbbb"`).

**Ví dụ 2:** `s = "bbaaaaabb"` → `2`. Cách duy nhất: xóa hai ký tự đầu.

**Ràng buộc:**
- `1 ≤ s.length ≤ 10⁵`
- `s[i]` là `'a'` hoặc `'b'`.

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [("aababbab", 2), ("bbaaaaabb", 2)]

def is_balanced(t):
    """Cân bằng <=> không có 'b' nào đứng trước 'a' <=> không có 'ba' liền kề (giải thích ngay dưới)."""
    return "ba" not in t

def delete_positions(s, pos):
    return ''.join(c for i, c in enumerate(s) if i not in set(pos))

for s, exp in examples:
    print(f"s={s!r}: cân bằng? {is_balanced(s)} -> kỳ vọng xóa {exp}")
print("Xóa {2, 6} khỏi aababbab:", delete_positions("aababbab", [2, 6]), is_balanced(delete_positions("aababbab", [2, 6])))
print("Xóa {3, 6} khỏi aababbab:", delete_positions("aababbab", [3, 6]), is_balanced(delete_positions("aababbab", [3, 6])))

s='aababbab': cân bằng? False -> kỳ vọng xóa 2
s='bbaaaaabb': cân bằng? False -> kỳ vọng xóa 2
Xóa {2, 6} khỏi aababbab: aaabbb True
Xóa {3, 6} khỏi aababbab: aabbbb True


**Vì sao kiểm "không có `ba` liền kề" là đủ?** Nếu tồn tại `i < j` với `s[i] = 'b'`, `s[j] = 'a'`, thì đi từ `i` tới `j` phải có một chỗ chuyển từ `b` sang `a` **liền kề**. Ngược lại, có `"ba"` liền kề thì hiển nhiên vi phạm. Vậy xâu cân bằng ⇔ không chứa xâu con `"ba"` ⇔ có dạng `a…ab…b`.

## 3. Kiến thức nền

### 3.1 Đếm ký tự: `str.count` và biểu thức boolean

`s.count('a')` đếm số `'a'` trong `O(n)`. Trong vòng lặp, `cnt += (c == 'a')` cộng `1` khi đúng (vì `True == 1`).

In [2]:
s = "aababbab"
print("số a:", s.count('a'), "| số b:", s.count('b'))
cnt = 0
for c in s:
    cnt += (c == 'b')
print("đếm b bằng boolean:", cnt)

số a: 4 | số b: 4
đếm b bằng boolean: 4


### 3.2 Mảng tiền tố và hậu tố

`pre_b[k]` = số `'b'` trong `s[:k]`; `suf_a[k]` = số `'a'` trong `s[k:]`. Cả hai dựng trong `O(n)`.

In [3]:
n = len(s)
pre_b = [0] * (n + 1)
for i, c in enumerate(s):
    pre_b[i + 1] = pre_b[i] + (c == 'b')
suf_a = [0] * (n + 1)
for i in range(n - 1, -1, -1):
    suf_a[i] = suf_a[i + 1] + (s[i] == 'a')
print("s     =", list(s))
print("pre_b =", pre_b)
print("suf_a =", suf_a)

s     = ['a', 'a', 'b', 'a', 'b', 'b', 'a', 'b']
pre_b = [0, 0, 0, 1, 1, 2, 3, 3, 4]
suf_a = [4, 3, 2, 2, 1, 1, 1, 0, 0]


### 3.3 Ngăn xếp bằng list

`stack.append(x)` đẩy vào, `stack.pop()` lấy ra phần tử trên cùng, `stack[-1]` xem đỉnh. Tất cả `O(1)`.

In [4]:
stack = []
for c in "abba":
    if stack and stack[-1] == 'b' and c == 'a':
        print(f"gặp '{c}', đỉnh là 'b' -> pop"); stack.pop()
    else:
        stack.append(c); print(f"đẩy '{c}': {stack}")

đẩy 'a': ['a']
đẩy 'b': ['a', 'b']
đẩy 'b': ['a', 'b', 'b']
gặp 'a', đỉnh là 'b' -> pop


### 3.4 `itertools.combinations` để liệt kê tập con

`combinations(range(n), k)` liệt kê mọi tập `k` chỉ số — dùng cho brute force "thử mọi cách xóa `k` ký tự", tăng dần `k`.

In [5]:
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 4. Ý tưởng nền tảng / trực giác

Xâu kết quả có dạng `a…ab…b`. Hãy tưởng tượng một **vạch cắt** ở giữa: mọi ký tự **bên trái** vạch phải là `a` (nên xóa mọi `b` bên trái), mọi ký tự **bên phải** phải là `b` (xóa mọi `a` bên phải). Chi phí của vạch cắt ở vị trí `k` (giữa `s[k−1]` và `s[k]`):

`cost(k) = (số 'b' trong s[:k]) + (số 'a' trong s[k:])`

Đáp án = `min` trên `k = 0, 1, …, n`.

Tính tay ví dụ 2 `s = "bbaaaaabb"` (`n = 9`):

| k | s[:k] | s[k:] | b bên trái | a bên phải | cost |
|---|---|---|---|---|---|
| 0 | "" | bbaaaaabb | 0 | 5 | 5 |
| 1 | b | baaaaabb | 1 | 5 | 6 |
| 2 | bb | aaaaabb | 2 | 5 | 7 |
| 7 | bbaaaaa | bb | 2 | 0 | **2** |
| 9 | bbaaaaabb | "" | 4 | 0 | 4 |

In [6]:
def show_costs(s):
    n = len(s)
    rows = []
    for k in range(n + 1):
        lb, ra = s[:k].count('b'), s[k:].count('a')
        rows.append((k, lb, ra, lb + ra))
    for k, lb, ra, c in rows:
        print(f"k={k}: '{s[:k]}' | '{s[k:]}'  -> b trái={lb}, a phải={ra}, cost={c}")
    print("min =", min(r[3] for r in rows))

show_costs("bbaaaaabb")

k=0: '' | 'bbaaaaabb'  -> b trái=0, a phải=5, cost=5
k=1: 'b' | 'baaaaabb'  -> b trái=1, a phải=5, cost=6
k=2: 'bb' | 'aaaaabb'  -> b trái=2, a phải=5, cost=7
k=3: 'bba' | 'aaaabb'  -> b trái=2, a phải=4, cost=6
k=4: 'bbaa' | 'aaabb'  -> b trái=2, a phải=3, cost=5
k=5: 'bbaaa' | 'aabb'  -> b trái=2, a phải=2, cost=4
k=6: 'bbaaaa' | 'abb'  -> b trái=2, a phải=1, cost=3
k=7: 'bbaaaaa' | 'bb'  -> b trái=2, a phải=0, cost=2
k=8: 'bbaaaaab' | 'b'  -> b trái=3, a phải=0, cost=3
k=9: 'bbaaaaabb' | ''  -> b trái=4, a phải=0, cost=4
min = 2


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn theo định nghĩa: thử xóa `k` ký tự, `k = 0, 1, 2, …`

Đúng theo đề nhưng `O(2ⁿ · n)` — chỉ dùng cho `n ≤ 15` để làm **chuẩn đối chiếu**.

In [7]:
def min_del_subsets(s):
    n = len(s)
    for k in range(n + 1):
        for pos in combinations(range(n), k):
            ps = set(pos)
            t = ''.join(c for i, c in enumerate(s) if i not in ps)
            if "ba" not in t:
                return k
    return n

for s, exp in examples:
    print(s, "->", min_del_subsets(s), "(kỳ vọng", exp, ")")

aababbab -> 2 (kỳ vọng 2 )
bbaaaaabb -> 2 (kỳ vọng 2 )


### 5.2 Thử mọi điểm cắt, mỗi điểm đếm lại từ đầu

Dùng nhận xét ở mục 4: chỉ cần thử `n + 1` điểm cắt, nhưng mỗi điểm đếm lại bằng `count` tốn `O(n)`.

In [8]:
def min_del_split_naive(s):
    n = len(s)
    return min(s[:k].count('b') + s[k:].count('a') for k in range(n + 1))

for s, exp in examples:
    print(s, "->", min_del_split_naive(s), "(kỳ vọng", exp, ")")

aababbab -> 2 (kỳ vọng 2 )
bbaaaaabb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** cách 5.1 `O(2ⁿ · n)`; cách 5.2 `O(n²)` thời gian (với `n = 10⁵` là `10¹⁰` — quá chậm), `O(n)` bộ nhớ cho các lát cắt.

**Nút thắt:** khi dịch vạch cắt từ `k` sang `k + 1`, chỉ **một ký tự** `s[k]` đổi bên — không cần đếm lại tất cả.

## 6. Ý tưởng thuật toán / bước đột phá

**Cập nhật gia tăng:** khi vạch cắt đi qua ký tự `c = s[k]` (từ bên phải sang bên trái):
- **Nếu** `c == 'a'` **thì** số `a` bên phải giảm 1 ⇒ `cost` giảm 1.
- **Nếu** `c == 'b'` **thì** số `b` bên trái tăng 1 ⇒ `cost` tăng 1.

⇒ Bắt đầu `cost(0) = tổng số 'a'`, rồi quét một lượt, mỗi bước `±1`, giữ `min`. `O(n)` thời gian, `O(1)` bộ nhớ.

**Góc nhìn quy hoạch động:** gọi `f[i]` = số xóa ít nhất để `s[:i]` cân bằng, `b_i` = số `b` trong `s[:i]`.
- **Nếu** `s[i−1] == 'b'` **thì** `f[i] = f[i−1]` (thêm `b` vào cuối xâu cân bằng vẫn cân bằng).
- **Nếu** `s[i−1] == 'a'` **thì** hai lựa chọn: **xóa** chữ `a` này (`f[i−1] + 1`) hoặc **giữ** nó — khi đó mọi `b` phía trước phải bị xóa (`b_{i−1}`). `f[i] = min(f[i−1] + 1, b_{i−1})`.

**Bất biến (hai biến đếm):** sau khi xử lý `k` ký tự, `lb` = số `b` trong `s[:k]`, `ra` = số `a` trong `s[k:]`, và `ans = min_{j ≤ k} cost(j)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố `b` và hậu tố `a`

In [9]:
def min_del_prefix(s):
    n = len(s)
    pre_b = [0] * (n + 1)
    for i, c in enumerate(s):
        pre_b[i + 1] = pre_b[i] + (c == 'b')
    suf_a = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf_a[i] = suf_a[i + 1] + (s[i] == 'a')
    return min(pre_b[k] + suf_a[k] for k in range(n + 1))

for s, exp in examples:
    print(s, "->", min_del_prefix(s), "(kỳ vọng", exp, ")")

aababbab -> 2 (kỳ vọng 2 )
bbaaaaabb -> 2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Hai biến đếm chạy (`lb`, `ra`)

In [10]:
def min_del_counters(s):
    lb, ra = 0, s.count('a')          # vạch cắt ở k = 0
    ans = lb + ra
    for c in s:                       # dịch vạch cắt qua c
        if c == 'a':
            ra -= 1
        else:
            lb += 1
        ans = min(ans, lb + ra)
    return ans

class SolutionCounters:
    def minimumDeletions(self, s: str) -> int:
        lb, ra = 0, s.count('a')
        ans = ra
        for c in s:
            if c == 'a':
                ra -= 1
            else:
                lb += 1
            if lb + ra < ans:
                ans = lb + ra
        return ans

for s, exp in examples:
    print(s, "->", min_del_counters(s), SolutionCounters().minimumDeletions(s), "(kỳ vọng", exp, ")")

aababbab -> 2 2 (kỳ vọng 2 )
bbaaaaabb -> 2 2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Quy hoạch động một biến

`dp` = `f[i]`, `b` = số `'b'` đã gặp. Gặp `'b'`: `b += 1` (dp giữ nguyên). Gặp `'a'`: `dp = min(dp + 1, b)`.

In [11]:
def min_del_dp_table(s):
    n = len(s)
    f = [0] * (n + 1)
    b = 0
    for i, c in enumerate(s, 1):
        if c == 'b':
            f[i] = f[i - 1]
            b += 1
        else:
            f[i] = min(f[i - 1] + 1, b)
    return f[n]

def min_del_dp(s):
    dp = b = 0
    for c in s:
        if c == 'b':
            b += 1
        else:
            dp = min(dp + 1, b)
    return dp

for s, exp in examples:
    print(s, "->", min_del_dp_table(s), min_del_dp(s), "(kỳ vọng", exp, ")")

aababbab -> 2 2 (kỳ vọng 2 )
bbaaaaabb -> 2 2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n)` thời gian; bản bảng `O(n)` bộ nhớ, bản một biến `O(1)`.

### 7.4 Cách 5 — Ngăn xếp triệt tiêu cặp `b…a`

Duyệt từ trái sang phải. Gặp `'a'` mà còn một `'b'` "chưa ghép" phía trước ⇒ ghép cặp `(b, a)` và **xóa một** trong hai (`ans += 1`), bỏ `b` đó khỏi ngăn xếp. Ngược lại, đẩy ký tự vào. Thực tế chỉ cần đếm số `b` chưa ghép.

In [12]:
def min_del_stack(s):
    stack = []
    ans = 0
    for c in s:
        if c == 'a' and stack and stack[-1] == 'b':
            stack.pop()
            ans += 1
        else:
            stack.append(c)
    return ans

def min_del_stack_counter(s):
    open_b = ans = 0
    for c in s:
        if c == 'b':
            open_b += 1
        elif open_b:
            open_b -= 1
            ans += 1
    return ans

for s, exp in examples:
    print(s, "->", min_del_stack(s), min_del_stack_counter(s), "(kỳ vọng", exp, ")")

aababbab -> 2 2 (kỳ vọng 2 )
bbaaaaabb -> 2 2 (kỳ vọng 2 )


**Vì sao đúng (trực giác):** mỗi cặp `(b, a)` được ghép là **rời nhau** và mỗi cặp là một vi phạm cần ít nhất 1 lần xóa ⇒ `ans` là **chặn dưới**. Ngược lại, có thể chứng minh số cặp ghép tham lam bằng đúng `f[n]` của DP (mục 10). Ngăn xếp chỉ chứa dạng `a…ab…b`, nên đỉnh là `'b'` ⇔ còn `b` chưa ghép.

**Độ phức tạp:** `O(n)` thời gian; bản dùng ngăn xếp `O(n)` bộ nhớ, bản đếm `O(1)`.

## 8. TRACE từng bước

### 8.1 Trace cách 3 (hai biến đếm) trên ví dụ 1 `"aababbab"`

`s.count('a') = 4` ⇒ ban đầu `lb = 0`, `ra = 4`, `ans = 4`.

| k (sau khi qua) | c | lb | ra | cost | ans |
|---|---|---|---|---|---|
| 1 | a | 0 | 3 | 3 | 3 |
| 2 | a | 0 | 2 | 2 | 2 |
| 3 | b | 1 | 2 | 3 | 2 |
| 4 | a | 1 | 1 | 2 | 2 |
| 5 | b | 2 | 1 | 3 | 2 |
| 6 | b | 3 | 1 | 4 | 2 |
| 7 | a | 3 | 0 | 3 | 2 |
| 8 | b | 4 | 0 | 4 | 2 |

In [13]:
def trace_counters(s):
    lb, ra = 0, s.count('a')
    ans = lb + ra
    print(f"k=0: lb={lb}, ra={ra}, cost={lb + ra}, ans={ans}")
    for k, c in enumerate(s, 1):
        if c == 'a':
            ra -= 1
        else:
            lb += 1
        ans = min(ans, lb + ra)
        print(f"k={k}: qua '{c}' -> lb={lb}, ra={ra}, cost={lb + ra}, ans={ans}   (trái='{s[:k]}', phải='{s[k:]}')")
    return ans

trace_counters("aababbab")

k=0: lb=0, ra=4, cost=4, ans=4
k=1: qua 'a' -> lb=0, ra=3, cost=3, ans=3   (trái='a', phải='ababbab')
k=2: qua 'a' -> lb=0, ra=2, cost=2, ans=2   (trái='aa', phải='babbab')
k=3: qua 'b' -> lb=1, ra=2, cost=3, ans=2   (trái='aab', phải='abbab')
k=4: qua 'a' -> lb=1, ra=1, cost=2, ans=2   (trái='aaba', phải='bbab')
k=5: qua 'b' -> lb=2, ra=1, cost=3, ans=2   (trái='aabab', phải='bab')
k=6: qua 'b' -> lb=3, ra=1, cost=4, ans=2   (trái='aababb', phải='ab')
k=7: qua 'a' -> lb=3, ra=0, cost=3, ans=2   (trái='aababba', phải='b')
k=8: qua 'b' -> lb=4, ra=0, cost=4, ans=2   (trái='aababbab', phải='')


2

### 8.2 Trace cách 4 (DP) và cách 5 (ngăn xếp) trên ví dụ 2 `"bbaaaaabb"`

In [14]:
def trace_dp(s):
    dp = b = 0
    for i, c in enumerate(s, 1):
        if c == 'b':
            b += 1
            print(f"i={i} '{c}': b={b}, dp giữ {dp}")
        else:
            old = dp
            dp = min(dp + 1, b)
            print(f"i={i} '{c}': dp = min(xóa a: {old}+1, xóa mọi b trước: {b}) = {dp}")
    return dp

def trace_stack(s):
    open_b = ans = 0
    for i, c in enumerate(s):
        if c == 'b':
            open_b += 1; msg = f"b chưa ghép = {open_b}"
        elif open_b:
            open_b -= 1; ans += 1; msg = f"ghép với một b -> ans={ans}, còn {open_b} b chưa ghép"
        else:
            msg = "a không có b phía trước, giữ"
        print(f"  {i}: '{c}' {msg}")
    return ans

print("DP:"); trace_dp("bbaaaaabb")
print("Ngăn xếp:"); trace_stack("bbaaaaabb")

DP:
i=1 'b': b=1, dp giữ 0
i=2 'b': b=2, dp giữ 0
i=3 'a': dp = min(xóa a: 0+1, xóa mọi b trước: 2) = 1
i=4 'a': dp = min(xóa a: 1+1, xóa mọi b trước: 2) = 2
i=5 'a': dp = min(xóa a: 2+1, xóa mọi b trước: 2) = 2
i=6 'a': dp = min(xóa a: 2+1, xóa mọi b trước: 2) = 2
i=7 'a': dp = min(xóa a: 2+1, xóa mọi b trước: 2) = 2
i=8 'b': b=3, dp giữ 2
i=9 'b': b=4, dp giữ 2
Ngăn xếp:
  0: 'b' b chưa ghép = 1
  1: 'b' b chưa ghép = 2
  2: 'a' ghép với một b -> ans=1, còn 1 b chưa ghép
  3: 'a' ghép với một b -> ans=2, còn 0 b chưa ghép
  4: 'a' a không có b phía trước, giữ
  5: 'a' a không có b phía trước, giữ
  6: 'a' a không có b phía trước, giữ
  7: 'b' b chưa ghép = 1
  8: 'b' b chưa ghép = 2


2

## 9. Code nộp LeetCode

Lời giải chính: quy hoạch động một biến (ngắn, `O(n)`/`O(1)`).

In [15]:
class Solution:
    def minimumDeletions(self, s: str) -> int:
        dp = 0          # số xóa ít nhất cho tiền tố đã duyệt
        b = 0           # số 'b' đã gặp
        for c in s:
            if c == 'b':
                b += 1
            else:
                dp = min(dp + 1, b)
        return dp

for s, exp in examples:
    assert Solution().minimumDeletions(s) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (điểm cắt):** xâu sau khi xóa là cân bằng ⇔ nó có dạng `aᵖ bᵠ`. Gọi `k` là số ký tự **gốc** đứng trước ký tự `b` đầu tiên được giữ (hoặc `k = n` nếu không giữ `b` nào). Mọi `b` ở vị trí `< k` đều bị xóa, mọi `a` ở vị trí `≥ k` đều bị xóa ⇒ số xóa `≥ cost(k)`. Ngược lại xóa đúng các ký tự đó cho xâu cân bằng với đúng `cost(k)` lần xóa. ⇒ Đáp án `= min_k cost(k)`. Cách 2, 3 tính đúng biểu thức này (cách 3 duy trì `lb`, `ra` bằng cập nhật `±1` khi vạch cắt dịch qua một ký tự).

**DP (cách 4):** định nghĩa `cost_i(k)` = (số `b` trong `s[:k]`) + (số `a` trong `s[k:i]`) là chi phí của điểm cắt `k` trên tiền tố `s[:i]`, và `f[i] = min_{k ≤ i} cost_i(k)`; gọi `b_i` = số `b` trong `s[:i]`.
- Nếu `s[i−1] = 'b'`: với `k ≤ i−1`, chữ `b` mới nằm bên phải ⇒ `cost_i(k) = cost_{i−1}(k)`; còn `cost_i(i) = b_i = b_{i−1} + 1 = cost_{i−1}(i−1) + 1 > f[i−1]`. Vậy `f[i] = f[i−1]`.
- Nếu `s[i−1] = 'a'`: với `k ≤ i−1`, chữ `a` mới nằm bên phải và phải xóa ⇒ `cost_i(k) = cost_{i−1}(k) + 1`; còn `cost_i(i) = b_{i−1}` (giữ chữ `a`, xóa mọi `b` trước). Vậy `f[i] = min(f[i−1] + 1, b_{i−1})` — đúng công thức trong code.

**Ngăn xếp (cách 5):** có thể kiểm tra bằng quy nạp rằng sau mỗi bước, `ans` (số cặp đã ghép) bằng `f[i]` và `open_b` bằng `b_i − f[i]`: gặp `b` thì `open_b` tăng, `f` không đổi; gặp `a`: nếu `open_b > 0` tức `b_{i−1} > f[i−1]`, nên `min(f[i−1]+1, b_{i−1}) = f[i−1]+1` ⇒ `ans` tăng 1, `open_b` giảm 1 (khớp `b_i − f[i]`); nếu `open_b = 0` thì `b_{i−1} = f[i−1]` ⇒ `f[i] = f[i−1]`, giữ nguyên. Ta còn kiểm thực nghiệm ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn tập con | `O(2ⁿ · n)` | `O(n)` |
| 1b. Điểm cắt đếm lại | `O(n²)` | `O(n)` |
| 2. Tiền tố/hậu tố | `O(n)` | `O(n)` |
| 3. Hai biến đếm | `O(n)` | `O(1)` |
| 4. DP một biến | `O(n)` | `O(1)` |
| 5. Ngăn xếp / đếm | `O(n)` | `O(n)` / `O(1)` |

Đo trên `n = 10⁵` (bản `O(n²)` chỉ chạy `n = 3000`):

In [16]:
rnd = random.Random(1653)
big = ''.join(rnd.choice("ab") for _ in range(10**5))
for name, fn in [("prefix", min_del_prefix), ("counters", min_del_counters), ("dp", min_del_dp), ("stack", min_del_stack_counter)]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<9}: {r:>6}  {dt * 1000:6.1f} ms")
t0 = time.perf_counter(); r = min_del_split_naive(big[:3000]); dt = time.perf_counter() - t0
print(f"naive n=3000: {r} ({dt * 1000:.0f} ms) | dp cùng input: {min_del_dp(big[:3000])}")

prefix   :  49959    16.9 ms
counters :  49959     6.0 ms
dp       :  49959     3.4 ms
stack    :  49959     3.0 ms
naive n=3000: 1465 (7 ms) | dp cùng input: 1465


## 12. Bộ test

In [17]:
ALL = {
    "split_naive": min_del_split_naive,
    "prefix": min_del_prefix,
    "counters": min_del_counters,
    "dp_table": min_del_dp_table,
    "dp": min_del_dp,
    "stack": min_del_stack,
    "stack_counter": min_del_stack_counter,
    "SolCounters": lambda s: SolutionCounters().minimumDeletions(s),
    "Solution": lambda s: Solution().minimumDeletions(s),
}

def check(s, exp=None, brute=True):
    ref = min_del_subsets(s) if brute else min_del_split_naive(s)
    for name, fn in ALL.items():
        assert fn(s) == ref, (name, s, fn(s), ref)
    if exp is not None:
        assert ref == exp, (s, ref, exp)
    return ref

for s, exp in examples:
    check(s, exp)
print("Ví dụ đề: OK")

for s, exp in [("a", 0), ("b", 0), ("ab", 0), ("ba", 1), ("aaa", 0), ("bbb", 0),
               ("bbbaaa", 3), ("bababa", 3), ("ababab", 2), ("abba", 1), ("baab", 1)]:
    check(s, exp)
print("Edge case: OK")

# mọi xâu độ dài <= 10 (tổng 2046 xâu)
from itertools import product
total = 0
for n in range(1, 11):
    for tup in product("ab", repeat=n):
        check(''.join(tup)); total += 1
print(f"Toàn bộ {total} xâu độ dài 1..10 so với vét cạn tập con: OK")

Ví dụ đề: OK
Edge case: OK


Toàn bộ 2046 xâu độ dài 1..10 so với vét cạn tập con: OK


In [18]:
rnd = random.Random(2020)
for _ in range(300):
    n = rnd.randint(1, 400)
    p = rnd.random()
    s = ''.join('a' if rnd.random() < p else 'b' for _ in range(n))
    check(s, brute=False)
print("300 xâu ngẫu nhiên độ dài tới 400 (so với điểm cắt O(n^2)): OK")

300 xâu ngẫu nhiên độ dài tới 400 (so với điểm cắt O(n^2)): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số lần xuất hiện của `"ba"` liền kề

In [19]:
def min_del_bug_count_ba(s):
    return s.count("ba")                   # SAI: số "ba" liền kề không phải số lần xóa

for s in ["bbaa", "bbaaaaabb"]:
    print(f"{s}: sai = {min_del_bug_count_ba(s)}, đúng = {Solution().minimumDeletions(s)}")

bbaa: sai = 1, đúng = 2
bbaaaaabb: sai = 1, đúng = 2


**Vì sao sai:** sau khi xóa một ký tự, các ký tự khác **trở nên liền kề** và tạo `"ba"` mới (`"bbaa"` → xóa 1 → `"baa"` vẫn sai). Vi phạm là giữa **mọi cặp** `b` đứng trước `a`, không chỉ cặp liền kề. **Sửa:** dùng điểm cắt / DP.

### Lỗi 2 — Chỉ thử điểm cắt bên trong (`1..n−1`), bỏ `k = 0` và `k = n`

In [20]:
def min_del_bug_inner(s):
    n = len(s)
    if n == 1:
        return 0
    return min(s[:k].count('b') + s[k:].count('a') for k in range(1, n))   # SAI: thiếu k=0, k=n

for s in ["ba", "baa", "bba"]:
    print(f"{s}: sai = {min_del_bug_inner(s)}, đúng = {Solution().minimumDeletions(s)}")

ba: sai = 2, đúng = 1
baa: sai = 2, đúng = 1
bba: sai = 2, đúng = 1


**Vì sao sai:** `k = 0` ⇔ "giữ toàn `b`" (xóa mọi `a`), `k = n` ⇔ "giữ toàn `a`". Với `"baa"` cách tốt nhất là xóa chữ `b` đầu để còn `"aa"` — ứng với vạch cắt ở **cuối** (`k = n = 3`), mà bản sai không xét. Hai đầu mút luôn phải được xét. **Sửa:** `range(n + 1)`.

### Lỗi 3 — Tham lam một chiều: "xóa mọi `a` sau `b` đầu tiên" hoặc "xóa mọi `b` trước `a` cuối cùng"

In [21]:
def min_del_bug_greedy(s):
    fb = s.find('b')
    la = s.rfind('a')
    opt1 = s[fb:].count('a') if fb != -1 else 0       # xóa mọi a sau b đầu tiên
    opt2 = s[:la].count('b') if la != -1 else 0       # xóa mọi b trước a cuối cùng
    return min(opt1, opt2)

rnd = random.Random(7)
found = None
while found is None:
    t = ''.join(rnd.choice("ab") for _ in range(rnd.randint(2, 10)))
    if min_del_bug_greedy(t) != Solution().minimumDeletions(t):
        found = t
print(f"phản ví dụ tìm được: {found!r}: tham lam = {min_del_bug_greedy(found)}, đúng = {Solution().minimumDeletions(found)}")
f = found
print("   các điểm cắt của phản ví dụ:", [f[:k].count('b') + f[k:].count('a') for k in range(len(f) + 1)])

phản ví dụ tìm được: 'baaaaaabba': tham lam = 3, đúng = 2
   các điểm cắt của phản ví dụ: [7, 8, 7, 6, 5, 4, 3, 2, 3, 4, 3]


**Vì sao sai:** hai phương án đó chỉ là **hai điểm cắt đặc biệt** (ngay trước `b` đầu tiên, ngay sau `a` cuối cùng). Điểm cắt tối ưu có thể nằm **ở giữa**. **Sửa:** xét **mọi** điểm cắt (cách 3) hoặc DP.

### Lỗi 4 — DP quên cập nhật biến đếm `b`, hoặc dùng `f[i−1]` thay vì `f[i−1] + 1`

In [22]:
def min_del_bug_dp(s):
    dp = b = 0
    for c in s:
        if c == 'b':
            b += 1
        else:
            dp = min(dp, b)                 # SAI: quên +1 khi xóa chữ a hiện tại
    return dp

for s in ["aababbab", "bab"]:
    print(f"{s}: sai = {min_del_bug_dp(s)}, đúng = {Solution().minimumDeletions(s)}")

aababbab: sai = 0, đúng = 2
bab: sai = 0, đúng = 1


**Vì sao sai:** lựa chọn "xóa chữ `a` này" tốn **thêm 1** so với `f[i−1]`. Quên `+1` thì `dp` không bao giờ tăng ⇒ luôn trả 0. **Sửa:** `dp = min(dp + 1, b)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Xâu nhị phân "sau khi xóa phải có dạng `0…01…1`" ⇒ **thử mọi điểm cắt** với tiền tố/hậu tố ⇒ một lượt quét `O(n)`.
- Tương đương: đáp án = `n − (độ dài dãy con không giảm dài nhất)`; với bảng chữ 2 ký tự, LNDS tính trong `O(n)`.
- Khi dịch vạch cắt qua một phần tử, chi phí thay đổi `±1` ⇒ giữ biến chạy, không cần mảng.
- DP "xóa phần tử hiện tại hay xóa mọi thứ xung đột phía trước" là khuôn chung (LeetCode 926 — Flip String to Monotone Increasing gần như giống hệt).
- Ngăn xếp "triệt tiêu cặp xung đột" cho lời giải gọn, nhưng nên kiểm lại bằng brute force.

## 15. Biến thể của bài

1. **LeetCode 926 — Flip String to Monotone Increasing:** **lật** `0/1` thay vì xóa — cùng công thức điểm cắt và cùng DP.
2. **Chi phí xóa khác nhau** (`cost_a`, `cost_b` hoặc theo vị trí): điểm cắt vẫn đúng: `cost(k) = Σ cost của b bên trái + Σ cost của a bên phải`.
3. **Ba ký tự `a < b < c`, muốn dạng `a…ab…bc…c`:** hai vạch cắt, hoặc DP 3 trạng thái (ký tự cuối được giữ).
4. **Dãy con không giảm dài nhất** của mảng tổng quát: `O(n log n)` (patience sorting); bài này là trường hợp 2 giá trị.
5. **Trả về chính các vị trí cần xóa:** ghi lại điểm cắt tốt nhất `k*`.

In [23]:
def min_flips_monotone(s):
    """LeetCode 926: số lần lật ít nhất để '0...01...1'. Cùng DP: gặp '0' thì lật nó hoặc lật mọi '1' trước."""
    dp = ones = 0
    for c in s:
        if c == '1':
            ones += 1
        else:
            dp = min(dp + 1, ones)
    return dp

def min_del_weighted(s, wa, wb):
    """Xóa 'a' tốn wa, xóa 'b' tốn wb."""
    lb, ra = 0, wa * s.count('a')
    ans = lb + ra
    for c in s:
        if c == 'a': ra -= wa
        else: lb += wb
        ans = min(ans, lb + ra)
    return ans

print("926 '00110' ->", min_flips_monotone("00110"), "| '010110' ->", min_flips_monotone("010110"))
print("có trọng số 'bbaaaaabb', xóa a tốn 1, b tốn 10 ->", min_del_weighted("bbaaaaabb", 1, 10))
assert all(min_del_weighted(''.join(t), 1, 1) == Solution().minimumDeletions(''.join(t)) for n in range(1, 9) for t in product("ab", repeat=n))

926 '00110' -> 1 | '010110' -> 2
có trọng số 'bbaaaaabb', xóa a tốn 1, b tốn 10 -> 5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Tập con | thử xóa k ký tự | `O(2ⁿ n)` | `O(n)` | chuẩn đối chiếu |
| 1b. Điểm cắt đếm lại | `min_k` (b trái + a phải) | `O(n²)` | `O(n)` | hiểu bài |
| 2. Tiền tố/hậu tố | mảng `pre_b`, `suf_a` | `O(n)` | `O(n)` | rõ ràng, dễ giải thích |
| 3. Hai biến đếm | dịch vạch cắt `±1` | `O(n)` | `O(1)` | tối ưu, dễ mở rộng có trọng số |
| 4. DP một biến | xóa `a` này hay xóa mọi `b` trước | `O(n)` | `O(1)` | **nộp bài**, ngắn |
| 5. Ngăn xếp | ghép cặp `b…a` | `O(n)` | `O(1)`/`O(n)` | trực giác "triệt tiêu" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumDeletions("babaab")` bằng bao nhiêu?

**Bài 2 (tính tay):** Lập bảng `k`, `lb`, `ra`, `cost` cho `s = "abbab"` và chỉ ra mọi điểm cắt tối ưu.

**Bài 3 (điền TODO):** Hoàn thiện `positions_to_delete(s)` trả về **danh sách chỉ số** cần xóa (một phương án tối ưu) và tự kiểm xâu còn lại cân bằng.

**Bài 4 (tự viết lại):** Viết lại lời giải bằng công thức `n − LNDS(s)` với `LNDS` là độ dài dãy con không giảm dài nhất, tính trong `O(n)` cho xâu `a/b`.

**Bài 5 (mini challenge):** Biến thể ba ký tự: `s` gồm `'a'`, `'b'`, `'c'`; cân bằng nghĩa là có dạng `a…ab…bc…c`. Viết `min_del_abc(s)` trong `O(n)` và kiểm bằng vét cạn.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def positions_to_delete(s):
    # TODO: tìm điểm cắt k tốt nhất bằng hai biến đếm; trả về [i < k với s[i]=='b'] + [i >= k với s[i]=='a']
    pass

# Bài 4 — TODO
def min_del_lnds(s):
    # TODO: best_a = số a đã gặp (dãy toàn a), best = độ dài LNDS kết thúc bằng b
    pass

# Bài 5 — TODO
def min_del_abc(s):
    # TODO: DP 3 trạng thái: ka, kb, kc = độ dài dãy con không giảm dài nhất kết thúc bằng a/b/c
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Có 3 `a`, 3 `b`.
- Mức 2: Thử vài điểm cắt: `k = 0` cost 3, `k = 6` cost 3.
- Mức 3: Điểm cắt giữa có cost nhỏ hơn không? Dùng DP để chắc chắn.

**Bài 2**
- Mức 1: Ban đầu `ra = 2`.
- Mức 2: Qua `a` thì `ra − 1`, qua `b` thì `lb + 1`.
- Mức 3: Ghi `cost` cho `k = 0..5`, lấy mọi `k` đạt min.

**Bài 3**
- Mức 1: Lưu `best_k` khi cập nhật `ans`.
- Mức 2: Chỉ số cần xóa: `b` bên trái `best_k`, `a` bên phải.
- Mức 3: Kiểm `len(result) == Solution().minimumDeletions(s)` và xâu còn lại không chứa `"ba"`.

**Bài 4**
- Mức 1: LNDS của xâu `a/b` = `max_k (số a trong s[:k] + số b trong s[k:])`.
- Mức 2: Hoặc DP: `la` = LNDS kết thúc bằng `a`, `lb` = kết thúc bằng `b`.
- Mức 3: Gặp `a`: `la += 1`; gặp `b`: `lb = max(la, lb) + 1`.

**Bài 5**
- Mức 1: Giữ dãy con không giảm dài nhất trên 3 ký tự.
- Mức 2: Gặp `c`: `kc = max(ka, kb, kc) + 1`; gặp `b`: `kb = max(ka, kb) + 1`; gặp `a`: `ka += 1`.
- Mức 3: Đáp án `n − max(ka, kb, kc)`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
print("Bài 1:", Solution().minimumDeletions("babaab"))
assert Solution().minimumDeletions("babaab") == min_del_subsets("babaab") == 2

# Bài 2
show_costs("abbab")

# Bài 3
def positions_to_delete_sol(s):
    lb, ra = 0, s.count('a')
    best, best_k = lb + ra, 0
    for k, c in enumerate(s, 1):
        if c == 'a': ra -= 1
        else: lb += 1
        if lb + ra < best:
            best, best_k = lb + ra, k
    return [i for i in range(best_k) if s[i] == 'b'] + [i for i in range(best_k, len(s)) if s[i] == 'a']

rnd = random.Random(3)
for _ in range(2000):
    s = ''.join(rnd.choice("ab") for _ in range(rnd.randint(1, 30)))
    pos = positions_to_delete_sol(s)
    rest = ''.join(c for i, c in enumerate(s) if i not in set(pos))
    assert len(pos) == Solution().minimumDeletions(s) and "ba" not in rest
print("Bài 3 OK | aababbab ->", positions_to_delete_sol("aababbab"))

Bài 1: 2
k=0: '' | 'abbab'  -> b trái=0, a phải=2, cost=2
k=1: 'a' | 'bbab'  -> b trái=0, a phải=1, cost=1
k=2: 'ab' | 'bab'  -> b trái=1, a phải=1, cost=2
k=3: 'abb' | 'ab'  -> b trái=2, a phải=1, cost=3
k=4: 'abba' | 'b'  -> b trái=2, a phải=0, cost=2
k=5: 'abbab' | ''  -> b trái=3, a phải=0, cost=3
min = 1
Bài 3 OK | aababbab -> [3, 6]


In [26]:
# Bài 4
def min_del_lnds_sol(s):
    la = lb = 0                 # LNDS kết thúc bằng 'a' / bằng 'b'
    for c in s:
        if c == 'a':
            la += 1
        else:
            lb = max(la, lb) + 1
    return len(s) - max(la, lb)

for n in range(1, 11):
    for t in product("ab", repeat=n):
        t = ''.join(t)
        assert min_del_lnds_sol(t) == Solution().minimumDeletions(t)
print("Bài 4 OK")

# Bài 5
def min_del_abc_sol(s):
    ka = kb = kc = 0
    for ch in s:
        if ch == 'a':
            ka += 1
        elif ch == 'b':
            kb = max(ka, kb) + 1
        else:
            kc = max(ka, kb, kc) + 1
    return len(s) - max(ka, kb, kc)

def min_del_abc_brute(s):
    n = len(s)
    for k in range(n + 1):
        for pos in combinations(range(n), k):
            ps = set(pos)
            t = ''.join(c for i, c in enumerate(s) if i not in ps)
            if list(t) == sorted(t):
                return k
    return n

rnd = random.Random(5)
for _ in range(600):
    s = ''.join(rnd.choice("abc") for _ in range(rnd.randint(1, 9)))
    assert min_del_abc_sol(s) == min_del_abc_brute(s), s
print("Bài 5 OK | 'cbacba' ->", min_del_abc_sol("cbacba"))

Bài 4 OK
Bài 5 OK | 'cbacba' -> 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template điểm cắt "trái phải là A, phải phải là B":**

```python
left_bad, right_bad = 0, s.count(BAD_ON_RIGHT)     # vạch cắt ở 0
ans = left_bad + right_bad
for c in s:
    if c == BAD_ON_RIGHT: right_bad -= 1          # c chuyển sang trái: hết vi phạm bên phải
    else:                 left_bad += 1           # c (BAD_ON_LEFT) chuyển sang trái: thêm vi phạm
    ans = min(ans, left_bad + right_bad)
return ans
```

**Template DP:** `if c == 'b': b += 1 else: dp = min(dp + 1, b)`.

**Checklist:**
- [ ] Xét đủ `n + 1` điểm cắt (gồm `k = 0` và `k = n`)?
- [ ] Không nhầm "số `ba` liền kề" với số lần xóa?
- [ ] DP: lựa chọn "xóa `a` này" là `dp + 1`?
- [ ] Đếm `b` **trước** khi xử lý `a` hiện tại (thứ tự đúng)?
- [ ] Đã kiểm toàn bộ xâu độ dài ≤ 10 với vét cạn?
- [ ] Chữ ký đúng: `def minimumDeletions(self, s: str) -> int`?